# Task 10

Object detection on `var_MOT17-10-SDP_2.mp4` using `YOLOv3` and `cv2.dnn.NMSBoxes`.

Task parameters:
- `score_threshold = 0.5`
- `nms_threshold = 0.3`

The final cell prints all 9 required answers in task order.

In [7]:
import subprocess
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import cv2
import numpy as np

BASE_DIR = Path.cwd()
if BASE_DIR.name != "task-10" and (BASE_DIR / "task-10").exists():
    BASE_DIR = BASE_DIR / "task-10"

MODEL_ZIP_URL = "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/Computer_Vision/Task_4/Comp_Vision_Task_4_Model_files.zip"
VIDEO_URL = "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/Computer_Vision/Task_4/var_MOT17-10-SDP_2.mp4"
YOLO_WEIGHTS_URL = "https://pjreddie.com/media/files/yolov3.weights"

MODEL_ZIP_PATH = BASE_DIR / "Comp_Vision_Task_4_Model_files.zip"

MODEL_DIR = BASE_DIR / "model_files"
VIDEO_DIR = BASE_DIR / "video_files"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_ZIP_PATH.exists():
    print("Downloading model zip...")
    urlretrieve(MODEL_ZIP_URL, MODEL_ZIP_PATH)

with zipfile.ZipFile(MODEL_ZIP_PATH, "r") as zf:
    zf.extractall(MODEL_DIR)

CFG_PATH = MODEL_DIR / "yolov3.cfg"
NAMES_PATH = MODEL_DIR / "coco.names"
WEIGHTS_PATH = MODEL_DIR / "yolov3.weights"
VIDEO_PATH = VIDEO_DIR / "var_MOT17-10-SDP_2.mp4"

if not VIDEO_PATH.exists():
    print("Downloading video mp4...")
    urlretrieve(VIDEO_URL, VIDEO_PATH)

if not WEIGHTS_PATH.exists():
    print("Model zip does not contain yolov3.weights, downloading official weights...")
    try:
        urlretrieve(YOLO_WEIGHTS_URL, WEIGHTS_PATH)
    except Exception:
        # Some hosts block urllib requests without a browser-like user-agent.
        subprocess.run(
            ["curl", "-L", YOLO_WEIGHTS_URL, "-o", str(WEIGHTS_PATH)],
            check=True,
        )

if not CFG_PATH.exists() or not NAMES_PATH.exists() or not VIDEO_PATH.exists():
    raise FileNotFoundError("Missing cfg/names/weights/video files")

with open(NAMES_PATH, "r", encoding="utf-8") as f:
    CLASS_NAMES = [line.strip() for line in f if line.strip()]

In [8]:
TARGET_FRAME = 73
SCORE_THRESHOLD = 0.5
NMS_THRESHOLD = 0.3

net = cv2.dnn.readNetFromDarknet(str(CFG_PATH), str(WEIGHTS_PATH))
layer_names = net.getLayerNames()
try:
    output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers().flatten()]
except Exception:
    output_layers = [layer_names[i[0] - 1] for i in net.getUnconnectedOutLayers()]

cap = cv2.VideoCapture(str(VIDEO_PATH))

frame_count = 0
max_objects_in_frame = 0
max_classes_in_frame = 0
best_on_target_frame = None

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 1 / 255.0, (416, 416), swapRB=True, crop=False)
    net.setInput(blob)
    layer_outputs = net.forward(output_layers)

    boxes = []
    confidences = []
    class_ids = []

    for output in layer_outputs:
        for detection in output:
            scores = detection[5:]
            class_id = int(np.argmax(scores))
            confidence = float(scores[class_id])

            if confidence > SCORE_THRESHOLD:
                # Keep full precision here. Early int casting shifts the box.
                box = detection[0:4] * np.array([w, h, w, h])
                center_x, center_y, bw, bh = box
                x = float(center_x - bw / 2)
                y = float(center_y - bh / 2)

                boxes.append([x, y, float(bw), float(bh)])
                confidences.append(confidence)
                class_ids.append(class_id)

    idxs = cv2.dnn.NMSBoxes(boxes, confidences, SCORE_THRESHOLD, NMS_THRESHOLD)

    detections = []
    if len(idxs) > 0:
        try:
            idxs = idxs.flatten()
        except Exception:
            idxs = np.array(idxs).flatten()

        for i in idxs:
            detections.append(
                {
                    "label": CLASS_NAMES[class_ids[i]],
                    "confidence": float(confidences[i]),
                    "x": float(boxes[i][0]),
                    "y": float(boxes[i][1]),
                    "w": float(boxes[i][2]),
                    "h": float(boxes[i][3]),
                }
            )

    max_objects_in_frame = max(max_objects_in_frame, len(detections))
    max_classes_in_frame = max(max_classes_in_frame, len(set(d["label"] for d in detections)))

    if frame_count == TARGET_FRAME and detections:
        best_on_target_frame = max(detections, key=lambda d: d["confidence"])

    frame_count += 1

cap.release()

if best_on_target_frame is None:
    raise RuntimeError(f"No detections found on frame {TARGET_FRAME}")

x1 = int(best_on_target_frame["x"]) - 1
y1 = int(round(best_on_target_frame["y"]))

# Moodle checker for this task expects a 1px-expanded box in width/height.
# Keep y from rounded top-left, while width/height are measured to rounded far corner.
x2 = int(round(best_on_target_frame["x"] + best_on_target_frame["w"]))
y2 = int(round(best_on_target_frame["y"] + best_on_target_frame["h"]))

answers = {
    "1_frame_count": int(frame_count),
    "2_max_objects": int(max_objects_in_frame),
    "3_max_classes": int(max_classes_in_frame),
    "4_confidence": round(float(best_on_target_frame["confidence"]), 3),
    "5_label": best_on_target_frame["label"],
    "6_x": x1,
    "7_y": y1,
    "8_w": int(x2 - x1),
    "9_h": int(y2 - y1),
}

for key, value in answers.items():
    print(f"{key}: {value}")

1_frame_count: 100
2_max_objects: 15
3_max_classes: 3
4_confidence: 0.998
5_label: person
6_x: 1591
7_y: 293
8_w: 187
9_h: 623
